# Seismic Time-to-Depth Conversion Using a Velocity Cube

---

## 1. Theory

### 1.1 Why Convert from Time to Depth?

Seismic data is acquired and processed in the **two-way travel time (TWT)** domain — the time it takes for a sound wave to travel from the surface down to a reflector and back up. However, geologists and reservoir engineers need data in the **depth domain** (meters or feet) to:
- Correlate with well logs
- Build accurate geological models
- Calculate true structural relief and reservoir thickness

---

### 1.2 Fundamental Relationship

The relationship between depth $z$ and two-way travel time $t$ is governed by the seismic velocity $V$:

$$z = \int_0^{t/2} V(\tau) \, d\tau$$

where $\tau$ is the one-way travel time. For a **discrete, layered** velocity model with layers of interval velocity $V_i$ and one-way time thickness $\Delta t_i$:

$$z = \sum_{i=1}^{N} V_i \cdot \Delta t_i$$

Or equivalently, using the **average velocity** $\bar{V}$ at a given TWT $t$:

$$z = \bar{V}(t) \cdot \frac{t}{2}$$

---

### 1.3 Types of Velocity

| Velocity Type | Symbol | Definition | Use |
|---|---|---|---|
| **Interval Velocity** | $V_{int}$ | Velocity in a specific layer | Layer-by-layer depth conversion |
| **Average Velocity** | $\bar{V}$ | Average from surface to depth $z$ | Simple $z = \bar{V} \cdot t/2$ |
| **RMS Velocity** | $V_{rms}$ | Root-mean-square of interval velocities | NMO correction output |
| **Stacking Velocity** | $V_{stack}$ | Best-fit velocity for NMO | Approximates $V_{rms}$ |

The Dix equation converts RMS velocities to interval velocities:

$$V_{int}^2 = \frac{V_{rms,2}^2 \cdot t_2 - V_{rms,1}^2 \cdot t_1}{t_2 - t_1}$$

---

### 1.4 The Velocity Cube

A **velocity cube** is a 3D array $V(x, y, t)$ where:
- $x$, $y$ are inline/crossline (or spatial) coordinates
- $t$ is two-way travel time
- Each cell stores the **interval velocity** (or average/RMS velocity) at that location and time

---

### 1.5 Time-to-Depth Conversion Algorithm

For each trace $(x, y)$:

1. Extract the velocity profile $V(t)$ from the velocity cube at $(x, y)$
2. Compute the **cumulative depth** as a function of time:
   $$z(t_n) = \frac{1}{2} \sum_{i=1}^{n} V(t_i) \cdot \Delta t$$
   (factor of $\frac{1}{2}$ because $t$ is two-way time)
3. This gives a mapping $t \rightarrow z(t)$
4. **Resample** the seismic trace amplitude from regular time samples to regular depth samples using interpolation

The resampling step is critical because even though both the time and depth axes start at 0, the mapping $t \rightarrow z$ is **non-linear** (varies with laterally varying velocities).

---

### 1.6 Key Challenges

- **Velocity uncertainty**: Errors in velocity propagate directly into depth errors
- **Lateral velocity variation**: Requires trace-by-trace conversion (no single 1D model)
- **Interpolation artifacts**: Must use appropriate interpolation (linear, cubic, sinc)
- **Depth sampling**: Choose depth step $\Delta z$ carefully to avoid aliasing

---

## 2. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

print("✅ All imports successful")

## 3. Synthetic Seismic Data Generation

We create a realistic 3D synthetic dataset:
- **Seismic cube**: 3D array `(n_inline, n_crossline, n_time_samples)` with reflections from layered geology
- **Velocity cube**: Laterally varying interval velocities with depth trends

In [ ]:
# ── Grid parameters ──────────────────────────────────────────────────────────
n_inline     = 50        # number of inlines
n_crossline  = 60        # number of crosslines
n_time       = 401       # number of time samples
dt           = 0.002     # time sample interval [s]  → 2 ms
t_max        = (n_time - 1) * dt   # max TWT [s]

t_axis = np.linspace(0, t_max, n_time)  # TWT axis [s]

print(f"Seismic grid  : {n_inline} IL × {n_crossline} XL × {n_time} samples")
print(f"Time axis     : 0 → {t_max*1000:.0f} ms  (dt = {dt*1000:.0f} ms)")

# ── Ricker wavelet ────────────────────────────────────────────────────────────
def ricker(f, dt, duration=0.128):
    """Generate a zero-phase Ricker wavelet."""
    t_w = np.arange(-duration/2, duration/2 + dt, dt)
    pf  = (np.pi * f * t_w) ** 2
    w   = (1 - 2*pf) * np.exp(-pf)
    return t_w, w

t_w, wavelet = ricker(f=30, dt=dt)   # 30 Hz dominant frequency

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(t_w*1000, wavelet, 'steelblue', lw=2)
ax.set(xlabel='Time (ms)', ylabel='Amplitude', title='30 Hz Ricker Wavelet')
ax.axhline(0, color='k', lw=0.7, ls='--')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Reflectivity model ────────────────────────────────────────────────────────
# Define 5 horizons that dip slightly across IL/XL (simulates structural relief)

np.random.seed(42)
il_idx = np.arange(n_inline)
xl_idx = np.arange(n_crossline)
IL, XL = np.meshgrid(il_idx, xl_idx, indexing='ij')   # (n_IL, n_XL)

# Horizon TWT [s] — each horizon has a gentle dip + dome structure
horizon_centers = [0.20, 0.35, 0.50, 0.65, 0.75]  # base TWT at grid centre
reflectivities  = [0.08, -0.06, 0.10, -0.05, 0.07]

def make_horizon(t0, IL, XL, amp_dip=0.01, amp_dome=0.02):
    """Create a spatially varying horizon with dip and dome."""
    ni, nx = IL.shape
    dip  = amp_dip  * (IL / ni + XL / nx)
    dome = amp_dome * np.exp(-((IL - ni/2)**2 + (XL - nx/2)**2) / (ni*nx*0.05))
    return t0 + dip - dome

# Build reflectivity cube  (n_IL, n_XL, n_t)
refl_cube = np.zeros((n_inline, n_crossline, n_time))

for t0, rc in zip(horizon_centers, reflectivities):
    horizon_map = make_horizon(t0, IL, XL)
    for i in range(n_inline):
        for j in range(n_crossline):
            t_idx = int(round(horizon_map[i, j] / dt))
            if 0 <= t_idx < n_time:
                refl_cube[i, j, t_idx] += rc

# ── Convolve with wavelet to make seismic ────────────────────────────────────
from scipy.signal import fftconvolve

seismic_cube = np.zeros_like(refl_cube)
for i in range(n_inline):
    for j in range(n_crossline):
        seismic_cube[i, j, :] = np.convolve(refl_cube[i, j, :], wavelet, mode='same')

# Add mild random noise
seismic_cube += 0.002 * np.random.randn(*seismic_cube.shape)

print(f"Seismic cube shape : {seismic_cube.shape}  (IL, XL, t)")
print(f"Amplitude range    : {seismic_cube.min():.4f}  →  {seismic_cube.max():.4f}")

In [ ]:
# ── Velocity cube ─────────────────────────────────────────────────────────────
# Interval velocity increases with depth (compaction trend)
# Also varies laterally: higher velocities in the centre of the grid (e.g. carbonate body)

V_surface   = 1800.0   # m/s at t=0
V_gradient  = 600.0    # m/s per second of TWT (depth trend)
V_anomaly   = 300.0    # m/s lateral anomaly amplitude

# Base velocity: linear increase with TWT
V_base = V_surface + V_gradient * t_axis   # shape (n_t,)

# Lateral anomaly (Gaussian high-velocity body in centre)
lateral_anomaly = V_anomaly * np.exp(
    -((IL - n_inline/2)**2 + (XL - n_crossline/2)**2) / (n_inline * n_crossline * 0.08)
)  # shape (n_IL, n_XL)

# Full 3D velocity cube
vel_cube = V_base[np.newaxis, np.newaxis, :] + lateral_anomaly[:, :, np.newaxis]
vel_cube = gaussian_filter(vel_cube, sigma=[2, 2, 5])  # smooth it spatially & temporally

print(f"Velocity cube shape : {vel_cube.shape}  (IL, XL, t)")
print(f"Velocity range      : {vel_cube.min():.0f}  →  {vel_cube.max():.0f}  m/s")

## 4. Visualise the Input Data

In [ ]:
# ── Inline section (time domain) + velocity ───────────────────────────────────
il_plot = n_inline // 2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Seismic section
vm = np.percentile(np.abs(seismic_cube[il_plot]), 98)
im0 = axes[0].imshow(
    seismic_cube[il_plot].T,
    aspect='auto', cmap='seismic',
    vmin=-vm, vmax=vm,
    extent=[0, n_crossline, t_max*1000, 0]
)
axes[0].set(xlabel='Crossline', ylabel='TWT (ms)',
            title=f'Seismic — Inline {il_plot} (Time Domain)')
plt.colorbar(im0, ax=axes[0], label='Amplitude')

# Velocity section
im1 = axes[1].imshow(
    vel_cube[il_plot].T,
    aspect='auto', cmap='jet',
    extent=[0, n_crossline, t_max*1000, 0]
)
axes[1].set(xlabel='Crossline', ylabel='TWT (ms)',
            title=f'Interval Velocity — Inline {il_plot}')
plt.colorbar(im1, ax=axes[1], label='Velocity (m/s)')

plt.suptitle('Input Data — Time Domain', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Single trace + velocity profile ──────────────────────────────────────────
il_t, xl_t = n_inline // 2, n_crossline // 2

fig, axes = plt.subplots(1, 2, figsize=(8, 6), sharey=True)

axes[0].plot(seismic_cube[il_t, xl_t, :], t_axis*1000, 'steelblue', lw=1)
axes[0].invert_yaxis()
axes[0].set(xlabel='Amplitude', ylabel='TWT (ms)', title='Seismic Trace')
axes[0].axvline(0, color='k', lw=0.7)
axes[0].grid(True, alpha=0.3)

axes[1].plot(vel_cube[il_t, xl_t, :], t_axis*1000, 'orangered', lw=1.5)
axes[1].set(xlabel='Interval Velocity (m/s)', title='Velocity Profile')
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'IL={il_t}, XL={xl_t}', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Core Algorithm — Depth Function Computation

For each trace we:
1. Integrate the interval velocity over one-way time to get $z(t)$
2. Store the time→depth mapping for that trace

In [ ]:
def compute_depth_from_velocity(v_profile, dt):
    """
    Convert interval velocity profile to cumulative depth vs TWT.

    Parameters
    ----------
    v_profile : array (n_t,)  — interval velocity [m/s] at each TWT sample
    dt        : float         — TWT sample interval [s]

    Returns
    -------
    z_of_t : array (n_t,)  — depth [m] at each TWT sample

    Notes
    -----
    z(t_n) = (dt/2) * sum_{i=0}^{n} V_int(t_i)
    The dt/2 accounts for two-way travel time → one-way depth conversion.
    """
    dz = v_profile * (dt / 2.0)       # depth increment per TWT sample
    z_of_t = np.cumsum(dz)            # cumulative depth
    return z_of_t


# ── Demo on a single trace ────────────────────────────────────────────────────
v_demo  = vel_cube[il_t, xl_t, :]
z_of_t  = compute_depth_from_velocity(v_demo, dt)

print(f"Max TWT   : {t_axis[-1]*1000:.0f} ms")
print(f"Max depth : {z_of_t[-1]:.0f} m")

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(t_axis*1000, z_of_t, 'darkorchid', lw=2)
ax.set(xlabel='TWT (ms)', ylabel='Depth (m)', title='Time → Depth Mapping (single trace)')
ax.invert_yaxis()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Resampling — Time Domain → Depth Domain

Once we have $z(t)$, we interpolate the seismic amplitude trace (known at regular $t$ samples) onto a regular depth grid.

In [ ]:
def resample_trace_to_depth(seis_trace, z_of_t, z_axis, method='linear'):
    """
    Resample a seismic trace from time domain to depth domain.

    Parameters
    ----------
    seis_trace : array (n_t,)  — seismic amplitude at each TWT sample
    z_of_t     : array (n_t,)  — depth [m] at each TWT sample (from compute_depth_from_velocity)
    z_axis     : array (n_z,)  — desired output depth samples [m]
    method     : str           — interpolation method ('linear', 'cubic')

    Returns
    -------
    depth_trace : array (n_z,)  — seismic amplitude at each depth sample
    """
    # Build interpolator:  depth → amplitude
    # z_of_t must be strictly increasing for interpolation
    # Guard against duplicate z values (constant-velocity zones)
    _, unique_idx = np.unique(z_of_t, return_index=True)
    z_u = z_of_t[unique_idx]
    s_u = seis_trace[unique_idx]

    f_interp = interp1d(z_u, s_u, kind=method,
                        bounds_error=False, fill_value=0.0)
    return f_interp(z_axis)


# ── Define the output depth axis ──────────────────────────────────────────────
dz      = 4.0                                  # depth sample interval [m]
z_max   = z_of_t.max() * 1.05                  # add 5% buffer
z_axis  = np.arange(0, z_max, dz)
n_depth = len(z_axis)

print(f"Depth axis : 0 → {z_axis[-1]:.0f} m  (dz = {dz:.0f} m,  n_depth = {n_depth})")

# Demo on single trace
depth_trace_demo = resample_trace_to_depth(
    seismic_cube[il_t, xl_t, :], z_of_t, z_axis, method='linear'
)

fig, axes = plt.subplots(1, 2, figsize=(9, 6))

axes[0].plot(seismic_cube[il_t, xl_t, :], t_axis*1000, 'steelblue', lw=1)
axes[0].invert_yaxis()
axes[0].set(xlabel='Amplitude', ylabel='TWT (ms)', title='Original (Time Domain)')
axes[0].axvline(0, color='k', lw=0.7); axes[0].grid(True, alpha=0.3)

axes[1].plot(depth_trace_demo, z_axis, 'darkorange', lw=1)
axes[1].invert_yaxis()
axes[1].set(xlabel='Amplitude', ylabel='Depth (m)', title='Converted (Depth Domain)')
axes[1].axvline(0, color='k', lw=0.7); axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Single Trace Time→Depth  (IL={il_t}, XL={xl_t})', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Full 3D Time-to-Depth Conversion

In [ ]:
def time_to_depth_3d(seismic_cube, vel_cube, dt, dz=4.0,
                     interp_method='linear', verbose=True):
    """
    Convert a 3D seismic cube from time domain to depth domain.

    Parameters
    ----------
    seismic_cube  : ndarray (n_IL, n_XL, n_t)  — amplitude data in TWT
    vel_cube      : ndarray (n_IL, n_XL, n_t)  — interval velocity [m/s]
    dt            : float                       — TWT sample interval [s]
    dz            : float                       — output depth sample interval [m]
    interp_method : str                         — 'linear' or 'cubic'
    verbose       : bool                        — print progress

    Returns
    -------
    depth_cube : ndarray (n_IL, n_XL, n_z)
    z_axis     : ndarray (n_z,)  — depth axis [m]
    """
    n_il, n_xl, n_t = seismic_cube.shape

    # --- Step 1: Determine maximum depth across the entire cube ---------------
    z_max_global = 0.0
    for i in range(n_il):
        for j in range(n_xl):
            z_tmp = compute_depth_from_velocity(vel_cube[i, j, :], dt)
            z_max_global = max(z_max_global, z_tmp[-1])

    z_axis  = np.arange(0, z_max_global * 1.02, dz)
    n_z     = len(z_axis)
    depth_cube = np.zeros((n_il, n_xl, n_z), dtype=np.float32)

    if verbose:
        print(f"Output depth range : 0 → {z_axis[-1]:.0f} m  (n_z = {n_z})")

    # --- Step 2: Convert each trace -------------------------------------------
    for i in range(n_il):
        if verbose and i % 10 == 0:
            print(f"  Processing inline {i}/{n_il-1} ...", end='\r')
        for j in range(n_xl):
            z_of_t = compute_depth_from_velocity(vel_cube[i, j, :], dt)
            depth_cube[i, j, :] = resample_trace_to_depth(
                seismic_cube[i, j, :], z_of_t, z_axis, method=interp_method
            )

    if verbose:
        print(f"\n✅ Conversion complete. Depth cube shape: {depth_cube.shape}")

    return depth_cube, z_axis


# ── Run the conversion ─────────────────────────────────────────────────────────
depth_cube, z_axis = time_to_depth_3d(
    seismic_cube, vel_cube, dt, dz=4.0, interp_method='linear'
)

## 8. Results — Comparison of Time vs Depth Sections

In [ ]:
# ── Inline section comparison ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

vm = np.percentile(np.abs(seismic_cube[il_plot]), 98)

axes[0].imshow(
    seismic_cube[il_plot].T, aspect='auto', cmap='seismic',
    vmin=-vm, vmax=vm,
    extent=[0, n_crossline, t_max*1000, 0]
)
axes[0].set(xlabel='Crossline', ylabel='TWT (ms)',
            title=f'TIME Domain — Inline {il_plot}')

vm2 = np.percentile(np.abs(depth_cube[il_plot]), 98)
axes[1].imshow(
    depth_cube[il_plot].T, aspect='auto', cmap='seismic',
    vmin=-vm2, vmax=vm2,
    extent=[0, n_crossline, z_axis[-1], 0]
)
axes[1].set(xlabel='Crossline', ylabel='Depth (m)',
            title=f'DEPTH Domain — Inline {il_plot}')

plt.suptitle('Time ↔ Depth Domain Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Crossline section comparison ─────────────────────────────────────────────
xl_plot = n_crossline // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

vm = np.percentile(np.abs(seismic_cube[:, xl_plot, :]), 98)
axes[0].imshow(
    seismic_cube[:, xl_plot, :].T, aspect='auto', cmap='seismic',
    vmin=-vm, vmax=vm,
    extent=[0, n_inline, t_max*1000, 0]
)
axes[0].set(xlabel='Inline', ylabel='TWT (ms)',
            title=f'TIME Domain — Crossline {xl_plot}')

vm2 = np.percentile(np.abs(depth_cube[:, xl_plot, :]), 98)
axes[1].imshow(
    depth_cube[:, xl_plot, :].T, aspect='auto', cmap='seismic',
    vmin=-vm2, vmax=vm2,
    extent=[0, n_inline, z_axis[-1], 0]
)
axes[1].set(xlabel='Inline', ylabel='Depth (m)',
            title=f'DEPTH Domain — Crossline {xl_plot}')

plt.suptitle('Crossline Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Depth slices (time slice vs depth slice) ──────────────────────────────────
# Pick approximate equivalents
t_slice_ms   = 400   # ms TWT
z_slice_m    = 700   # m depth

t_idx = np.argmin(np.abs(t_axis*1000 - t_slice_ms))
z_idx = np.argmin(np.abs(z_axis - z_slice_m))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

vm = np.percentile(np.abs(seismic_cube[:, :, t_idx]), 98)
axes[0].imshow(
    seismic_cube[:, :, t_idx], aspect='auto', cmap='seismic',
    vmin=-vm, vmax=vm
)
axes[0].set(xlabel='Crossline', ylabel='Inline',
            title=f'Time Slice @ TWT = {t_slice_ms} ms')

vm2 = np.percentile(np.abs(depth_cube[:, :, z_idx]), 98) + 1e-9
axes[1].imshow(
    depth_cube[:, :, z_idx], aspect='auto', cmap='seismic',
    vmin=-vm2, vmax=vm2
)
axes[1].set(xlabel='Crossline', ylabel='Inline',
            title=f'Depth Slice @ {z_slice_m:.0f} m')

plt.suptitle('Horizontal Slices — Time vs Depth Domain', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Quantitative Analysis

### 9.1 Horizon Depth Error Analysis

We verify the conversion by checking where each horizon maps to in depth domain vs. the expected analytical depth.

In [ ]:
# ── Average velocity vs depth ──────────────────────────────────────────────────
# Compute average velocity across all traces at each TWT sample
avg_vel_profile = vel_cube.mean(axis=(0,1))    # (n_t,)

z_avg = compute_depth_from_velocity(avg_vel_profile, dt)

fig, axes = plt.subplots(1, 2, figsize=(10, 6))

axes[0].plot(avg_vel_profile, t_axis*1000, 'orangered', lw=2)
axes[0].invert_yaxis()
axes[0].set(xlabel='Avg Interval Velocity (m/s)', ylabel='TWT (ms)',
            title='Average Velocity Profile')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_axis*1000, z_avg, 'teal', lw=2)
axes[1].set(xlabel='TWT (ms)', ylabel='Depth (m)',
            title='Average Time → Depth Curve')
axes[1].grid(True, alpha=0.3)

# Mark horizons
for t0 in horizon_centers:
    t_idx_h = int(t0 / dt)
    z_h     = z_avg[min(t_idx_h, len(z_avg)-1)]
    axes[1].axhline(z_h, color='gray', lw=0.8, ls='--', alpha=0.7)
    axes[1].text(t_max*1000*0.05, z_h-15, f'{z_h:.0f} m', fontsize=8, color='gray')

plt.suptitle('Velocity & Time-Depth Relationship', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Lateral depth variation of deepest horizon ────────────────────────────────
# Show how the time-to-depth conversion warps the geometry

h_twt  = 0.65   # TWT of the 4th horizon [s]
h_tidx = int(h_twt / dt)

depth_map = np.zeros((n_inline, n_crossline))

for i in range(n_inline):
    for j in range(n_crossline):
        z_trace = compute_depth_from_velocity(vel_cube[i, j, :], dt)
        depth_map[i, j] = z_trace[h_tidx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im0 = axes[0].imshow(np.full((n_inline, n_crossline), h_twt*1000),
                     vmin=h_twt*1000-1, vmax=h_twt*1000+1,
                     cmap='plasma', aspect='auto')
axes[0].set(title=f'Horizon TWT = constant {h_twt*1000:.0f} ms',
            xlabel='Crossline', ylabel='Inline')
plt.colorbar(im0, ax=axes[0], label='TWT (ms)')

im1 = axes[1].imshow(depth_map, cmap='plasma', aspect='auto')
axes[1].set(title=f'Horizon Depth (varies laterally due to velocity)',
            xlabel='Crossline', ylabel='Inline')
plt.colorbar(im1, ax=axes[1], label='Depth (m)')

plt.suptitle('Lateral Velocity Variation Creates Depth Uncertainty', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Depth range for constant-TWT horizon: {depth_map.min():.1f} – {depth_map.max():.1f} m")
print(f"Depth variation                      : {depth_map.max() - depth_map.min():.1f} m")

## 10. Effect of Interpolation Method

In [ ]:
z_of_t_demo = compute_depth_from_velocity(vel_cube[il_t, xl_t, :], dt)
trace_demo  = seismic_cube[il_t, xl_t, :]

trace_linear = resample_trace_to_depth(trace_demo, z_of_t_demo, z_axis, method='linear')
trace_cubic  = resample_trace_to_depth(trace_demo, z_of_t_demo, z_axis, method='cubic')

fig, axes = plt.subplots(1, 3, figsize=(12, 6), sharey=True)

axes[0].plot(trace_demo, t_axis*1000, 'steelblue', lw=1)
axes[0].set(xlabel='Amplitude', ylabel='TWT (ms)', title='Original (Time)')
axes[0].invert_yaxis(); axes[0].axvline(0, color='k', lw=0.5)

axes[1].plot(trace_linear, z_axis, 'darkorange', lw=1)
axes[1].set(xlabel='Amplitude', title='Depth — Linear Interp')
axes[1].axvline(0, color='k', lw=0.5)

axes[2].plot(trace_cubic, z_axis, 'green', lw=1)
axes[2].set(xlabel='Amplitude', title='Depth — Cubic Interp')
axes[2].axvline(0, color='k', lw=0.5)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle('Interpolation Method Comparison', fontweight='bold')
plt.tight_layout()
plt.show()

diff = np.abs(trace_linear - trace_cubic)
print(f"Max absolute difference (linear vs cubic): {diff.max():.5f}")
print(f"Mean absolute difference                 : {diff.mean():.6f}")

## 11. Working with Real Data

### Using SEG-Y Files (industry standard format)

Real seismic data is stored in **SEG-Y format**. Use the `segyio` library.

In [ ]:
# ── Template for reading real SEG-Y data ─────────────────────────────────────
# Uncomment and adapt with your actual file paths

REAL_DATA_TEMPLATE = '''
import segyio
import numpy as np

# ── Load seismic cube ──────────────────────────────────────────────────────
with segyio.open("seismic_time.segy", iline=189, xline=193) as f:
    dt_real      = segyio.dt(f) * 1e-6          # sample interval in seconds
    seismic_cube = segyio.tools.cube(f)          # shape: (n_IL, n_XL, n_t)
    t_axis_real  = f.samples * 1e-3              # TWT in seconds
    n_il, n_xl, n_t = seismic_cube.shape

print(f"Seismic: {seismic_cube.shape}, dt = {dt_real*1000:.2f} ms")

# ── Load velocity cube (must be on same IL/XL/t grid as seismic) ───────────
with segyio.open("velocity_cube.segy", iline=189, xline=193) as f:
    vel_cube_real = segyio.tools.cube(f)         # shape: (n_IL, n_XL, n_t)

print(f"Velocity: {vel_cube_real.shape}")

# ── Convert ────────────────────────────────────────────────────────────────
depth_cube_real, z_axis_real = time_to_depth_3d(
    seismic_cube, vel_cube_real, dt_real, dz=5.0
)

# ── Save output as SEG-Y ───────────────────────────────────────────────────
spec = segyio.spec()
spec.sorting  = 2       # INLINE_3D
spec.format   = 1       # IBM float
spec.samples  = z_axis_real.astype(np.float32)
spec.ilines   = np.arange(n_il)
spec.xlines   = np.arange(n_xl)

with segyio.create("seismic_depth.segy", spec) as g:
    g.bin.update(tsort=segyio.TraceSortingFormat.INLINE_SORTING)
    for il_i, il in enumerate(spec.ilines):
        g.header.iline[il] = {segyio.TraceField.INLINE_3D: il}
        g.fast.raw[il_i]   = depth_cube_real[il_i]

print("✅ Depth domain SEG-Y saved to seismic_depth.segy")
'''

print("Real data template (SEG-Y):")
print("-" * 60)
print(REAL_DATA_TEMPLATE)

## 12. Summary & Key Takeaways

| Step | What we did | Key equation |
|---|---|---|
| 1 | Defined time and velocity grids | $V(x,y,t)$ |
| 2 | Computed depth from velocity | $z(t) = \frac{1}{2}\int_0^t V(\tau)d\tau$ |
| 3 | Resampled traces to uniform depth | Interpolation: $A(z_i)$ from $A(t_j)$ |
| 4 | Built 3D depth cube | Trace-by-trace conversion |

### Best Practices

- Use **interval velocity** for integration (not stacking velocity directly)
- Apply the **Dix equation** if only RMS velocities are available
- Choose **dz** small enough to avoid aliasing (typically $\leq V_{min} \cdot dt / 2$)
- **Cubic interpolation** is smoother but can overshoot near sharp impedance contrasts
- **Well ties** are essential to QC the velocity model before depth conversion
- Always check **frequency content preservation** after conversion

In [ ]:
# ── Final summary statistics ──────────────────────────────────────────────────
print("=" * 55)
print("   SEISMIC TIME-TO-DEPTH CONVERSION SUMMARY")
print("=" * 55)
print(f"Input  seismic cube  : {seismic_cube.shape}  (IL, XL, t)")
print(f"Input  velocity cube : {vel_cube.shape}   (IL, XL, t)")
print(f"Output depth cube    : {depth_cube.shape}  (IL, XL, z)")
print(f"")
print(f"Time axis   : 0 → {t_axis[-1]*1000:.0f} ms   (dt = {dt*1000:.0f} ms)")
print(f"Depth axis  : 0 → {z_axis[-1]:.0f} m  (dz = {dz:.0f} m)")
print(f"Velocity range : {vel_cube.min():.0f} – {vel_cube.max():.0f} m/s")
print("=" * 55)